Data Visualisation script for data categories

In [ ]:
# Mount Google Drive
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ✅ Set up dataset path
os.chdir("drive/My Drive/")
spreadsheet_path = "Auslan Static Dataset Breakdown"
dataset_path = "AuslanStaticData"

Mounted at /content/drive


In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
import pandas as pd

worksheet = gc.open('Auslan Static Dataset Breakdown').sheet1

# Get all values from the worksheet
rows = worksheet.get_all_values()

# Convert to DataFrame with first row as column names
spreadsheet = pd.DataFrame(rows[1:], columns=rows[0])

print(spreadsheet)

     ID Subject         Date            Device        Filename Class Lighting  \
0     0    Nick   Feb 8 2025  Dell XPS 15 9050    Nick_0_1.jpg     0  Natural   
1     1    Nick   Feb 8 2025  Dell XPS 15 9050    Nick_1_1.jpg     1  Natural   
2     2    Nick   Feb 8 2025  Dell XPS 15 9050    Nick_2_1.jpg     2  Natural   
3     3    Nick   Feb 8 2025  Dell XPS 15 9050    Nick_3_1.jpg     3  Natural   
4     4    Nick   Feb 8 2025  Dell XPS 15 9050    Nick_4_1.jpg     4  Natural   
...  ..     ...          ...               ...             ...   ...      ...   
4729       Nick  Apr 29 2025         iPhone 12  Nick_u_114.JPG     u  Natural   
4730       Nick  Apr 29 2025         iPhone 12  Nick_u_115.JPG     u  Natural   
4731       Nick  Apr 29 2025         iPhone 12  Nick_u_116.JPG     u  Natural   
4732       Nick  Apr 29 2025         iPhone 12  Nick_u_117.JPG     u  Natural   
4733       Nick  Apr 29 2025         iPhone 12  Nick_u_118.JPG     u  Natural   

     Light level           

In [ ]:
!sudo apt install cm-super dvipng texlive-latex-extra texlive-latex-recommended

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cm-super-minimal dvisvgm fonts-droid-fallback fonts-lato fonts-lmodern
  fonts-noto-mono fonts-texgyre fonts-urw-base35 ghostscript
  libapache-pom-java libcommons-logging-java libcommons-parent-java
  libfontbox-java libgs9 libgs9-common libidn12 libijs-0.35 libjbig2dec0
  libkpathsea6 libpdfbox-java libptexenc1 libruby3.0 libsynctex2 libteckit0
  libtexlua53 libtexluajit2 libwoff1 libzzip-0-13 lmodern pfb2t1c2pfb
  poppler-data preview-latex-style rake ruby ruby-net-telnet ruby-rubygems
  ruby-webrick ruby-xmlrpc ruby3.0 rubygems-integration t1utils tex-common
  tex-gyre texlive-base texlive-binaries texlive-fonts-recommended
  texlive-latex-base texlive-pictures texlive-plain-generic tipa
  xfonts-encodings xfonts-utils
Suggested packages:
  fonts-noto fonts-freefont-otf | fonts-freefont-ttf ghostscript-x
  libavalon-framework-java l

In [ ]:
# Get list of files
file_list = os.listdir(dataset_path)

# Convert to DataFrame
drive_files = pd.DataFrame(file_list, columns=['Filename'])

set1 = set(drive_files['Filename'])
set2 = set(spreadsheet["Filename"])

# Find missing files
missing_in_df2 = set1 - set2  # Files in df1 but not in df2
missing_in_df1 = set2 - set1  # Files in df2 but not in df1

print("Files missing in Spreadsheet:", missing_in_df2)
print("Files missing in Drive folder:", missing_in_df1)

print(set1)
print(set2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

# List of class categories to visualize
class_columns = [
    'Subject', 'Device', 'Light level', 'Lighting', 'Background',
    'Skin Tone', 'Distance', 'Orientation', 'Away from face',
    'Clothing', 'Hand'
]

# Activate LaTeX rendering
import matplotlib
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath,amssymb,amsfonts}'
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.serif'] = ['Computer Modern Roman']

# Set visual style
sns.set(style="whitegrid")

# Create a directory to store the images
if not os.path.exists('plots'):
  os.makedirs('plots')

# List of allowed classes (numbers and vowels)
allowed_classes = [str(i) for i in range(10)] + ['a', 'e', 'i', 'o', 'u']

# Anonymise signers
subject_mapping = {
     'Nick': 'A',
     'Evelyn': 'B',
     'Luke': 'C',
     'Daniel': 'D'
 }

# Filter the DataFrame to include only allowed classes
filtered_spreadsheet = spreadsheet[spreadsheet['Class'].isin(allowed_classes)]
filtered_spreadsheet['Subject'] = filtered_spreadsheet['Subject'].replace(subject_mapping)

# Create a directory to store the images
if not os.path.exists('plots'):
  os.makedirs('plots')

# Create subplots: one barplot per class and save as PNG
for i, col in enumerate(class_columns, 1):
    plt.figure(figsize=(12, 6))
    ax = sns.countplot(data=filtered_spreadsheet, x=col, palette="viridis", order=filtered_spreadsheet[col].value_counts().index)
    plt.title(f'Distribution of {col}', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.xlabel('')
    plt.ylabel('Count', fontsize=14)
    sns.despine(left=True)
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', label_type='edge', fontsize=12)

    # Save the plot as PNG
    plt.savefig(f'plots/{col}.png', bbox_inches='tight', dpi=300)
    plt.close()


<ipython-input-12-d05ef8ddb518>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_spreadsheet['Subject'] = filtered_spreadsheet['Subject'].replace(subject_mapping)
<ipython-input-12-d05ef8ddb518>:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(data=filtered_spreadsheet, x=col, palette="viridis", order=filtered_spreadsheet[col].value_counts().index)
<ipython-input-12-d05ef8ddb518>:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countp

In [ ]:
# Define the desired order of classes
class_order = [str(i) for i in range(10)] + ['a', 'e', 'i', 'o', 'u']

# Create the countplot with the specified order
plt.figure(figsize=(12, 6))
class_counts = filtered_spreadsheet['Class'].value_counts()
ax = sns.countplot(
    x=filtered_spreadsheet['Class'],
    order=class_order,  # Specify the desired order
    palette="viridis"
)

plt.title(
    'Number of Images per Class',
    fontsize=16  # Increase title font size
)
plt.xticks(rotation=0, ha='center', fontsize=12)  # Adjust x-axis tick labels
plt.xlabel('Class', fontsize=14)  # Increase x-axis label font size
plt.ylabel('Number of Images', fontsize=14)  # Increase y-axis label font size
# sns.despine(left=True)  # Remove left spine for a cleaner look
# ax.grid(False)  # Remove gridlines
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontsize=12)
plt.tight_layout()  # Adjust layout for better spacing
plt.savefig('plots/Number_of_Images_per_Class_Filtered.png', dpi=300)  # Increase DPI for higher resolution
plt.close()

<ipython-input-13-02b9cc7d1753>:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(


In [ ]:
# Prepare the data by grouping and counting
class_away_counts = filtered_spreadsheet.groupby(['Class', 'Away from face']).size().reset_index(name='count')

# Pivot the data for stacked plot
pivoted_counts = class_away_counts.pivot(index='Class', columns='Away from face', values='count').fillna(0)

# Ensure the classes are in the desired order
class_order = [str(i) for i in range(10)] + ['a', 'e', 'i', 'o', 'u']
pivoted_counts = pivoted_counts.reindex(class_order, fill_value=0)

# Create the stacked bar plot
plt.figure(figsize=(15, 8))
ax = pivoted_counts.plot(kind='bar', stacked=True, colormap='viridis', ax=plt.gca())

plt.title('Stacked Distribution of "Away from Face" by Class', fontsize=16)
plt.xlabel('Class', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(rotation=0)
plt.legend(title='Away from Face')
plt.tight_layout()

# Add value labels to the stacks
for container in ax.containers:
    labels = [f'{w:.0f}' if (w := v.get_height()) > 0 else '' for v in container]
    ax.bar_label(container, labels=labels, label_type='center', fontsize=10)

# Save the plot
plt.savefig('plots/stacked_away_from_face_by_class.png', dpi=300)
plt.close()